# EffTemp-Net Attention-Only — Held-Out Evaluation Only
## RWF-2000 + UCF-Crime Fighting subset + Bus Violence

Notebook ini **hanya melakukan inference/evaluation** menggunakan `best.pt`
yang sudah dilatih. Tidak ada training ulang.

Tidak ada:
- optimizer;
- scheduler;
- backpropagation;
- augmentation training;
- model selection menggunakan held-out data.

Arsitektur mengikuti notebook training
`EffTempNet_AttentionOnly_Seed123.ipynb`:

```text
16 cached frames
→ shared EfficientFormerV2-S0 backbone per frame
→ per-frame feature
→ feature dropout
→ single-layer bidirectional LSTM (hidden_size=32)
→ learned temporal attention
→ classifier 64 → 256 → 128 → 64 → 32 → 2
```

Temporal attention training yang dipertahankan adalah:

\[
u_t = \tanh(W h_t + b),
\]

\[
\alpha_t = \operatorname{softmax}(w^\top u_t),
\]

\[
c = \sum_{t=1}^{T}\alpha_t h_t.
\]

Tidak ada Sum/Mean/Max residual branch pada context akhir.

Ketiga held-out domain dievaluasi **terpisah**:
1. RWF-2000
2. UCF-Crime Fighting subset
3. Bus Violence

`predictions.csv` menyimpan prediksi clip-level dengan `sample_key` yang sama
dengan evaluator model lain, sehingga dapat dipakai untuk McNemar dan paired
bootstrap. Notebook juga menyimpan 16 bobot temporal-attention per clip.

## Mengganti seed

Setelah path dataset sudah benar, untuk seed lain **cukup ubah satu baris**:

```python
RUN_SEED = 123
```

menjadi `0` atau `42`.

Checkpoint dan folder output berubah otomatis.

In [1]:
# Jalankan sekali pada awal sesi Google Colab.
!pip -q install timm

In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
import os
import json
import random
import platform
import warnings
from contextlib import nullcontext
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import timm
from timm import create_model
from tqdm.auto import tqdm

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_recall_fscore_support,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    roc_curve,
    precision_recall_curve,
)

warnings.filterwarnings("ignore", category=UserWarning)

print("Python :", platform.python_version())
print("PyTorch:", torch.__version__)
print("timm   :", timm.__version__)
print("CUDA   :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU    :", torch.cuda.get_device_name(0))

Python : 3.13.15
PyTorch: 2.11.0+cu128
timm   : 1.0.28
CUDA   : True
GPU    : NVIDIA A100-SXM4-40GB


## 1. Konfigurasi

### Satu-satunya variabel yang diubah antar-seed

```python
RUN_SEED = 123
```

Pilihan yang valid:

```python
0
42
123
```

### Path held-out dataset

Default notebook mengasumsikan:

```text
/content/drive/MyDrive/IVS/ViolenceDataset/Benchmark/RWF
/content/drive/MyDrive/IVS/ViolenceDataset/Benchmark/UCF_Crime_Fighting
/content/drive/MyDrive/IVS/ViolenceDataset/Benchmark/BusViolence
```

Jika nama folder UCF atau Bus Violence berbeda, ubah `raw_dir` **sekali saja**.
Sesudah itu pergantian seed hanya membutuhkan perubahan `RUN_SEED`.

Cache UCF/Bus tetap shared dengan evaluator SingleFrame, FrameMean,
BiLSTM-Final, BiLSTM-Mean, TempSum, dan BiLSTM-Max agar seluruh model memakai
clip held-out yang identik.

In [4]:
# ============================================================
# SATU-SATUNYA VARIABEL YANG DIUBAH ANTAR-SEED
# ============================================================
RUN_SEED = 42   # <-- SATU-SATUNYA nilai yang diubah antar-seed: 0 / 42 / 123

if RUN_SEED not in (0, 42, 123):
    raise ValueError(
        f"RUN_SEED harus salah satu dari (0, 42, 123), mendapat {RUN_SEED}."
    )

# ============================================================
# MODEL / CHECKPOINT
# ============================================================
MODEL_ROOT = Path(
    "/content/drive/MyDrive/IVS/ViolenceDetection/Models/EffTempNet"
)

VARIANT = "attention_only"

# Candidate root mengikuti pola notebook training Attention-Only.
# Uploaded training notebook menggunakan:
# EffTempNet_AttentionOnly_NoLS_Seed123_v1/attention_only/seed_123/best.pt
CHECKPOINT_ROOT_CANDIDATES = [
    MODEL_ROOT / f"EffTempNet_AttentionOnly_NoLS_Seed{RUN_SEED}_v1",
    MODEL_ROOT / f"EffTempNet_AttentionOnly_NoLS_{RUN_SEED}Seed_v1",
    MODEL_ROOT / f"EffTempNet_AttentionOnly_NoLS_{RUN_SEED}Seeds_v1",
    MODEL_ROOT / "EffTempNet_AttentionOnly_NoLS_Seed0_v1",
    MODEL_ROOT / "EffTempNet_AttentionOnly_NoLS_Seed42_v1",
    MODEL_ROOT / "EffTempNet_AttentionOnly_NoLS_Seed123_v1",
]

# Optional manual override. Normalnya biarkan None.
CHECKPOINT_PATH_OVERRIDE = None

def resolve_checkpoint(seed: int) -> Path:
    if CHECKPOINT_PATH_OVERRIDE is not None:
        path = Path(CHECKPOINT_PATH_OVERRIDE)

        if not path.exists():
            raise FileNotFoundError(
                f"CHECKPOINT_PATH_OVERRIDE tidak ditemukan: {path}"
            )

        return path

    direct_matches = []

    for root in CHECKPOINT_ROOT_CANDIDATES:
        candidate = (
            root
            / VARIANT
            / f"seed_{seed}"
            / "best.pt"
        )

        if candidate.exists():
            direct_matches.append(candidate)

    direct_matches = list(
        dict.fromkeys(direct_matches)
    )

    if len(direct_matches) == 1:
        return direct_matches[0]

    if len(direct_matches) > 1:
        raise RuntimeError(
            "Ditemukan lebih dari satu candidate best.pt Attention-Only. "
            "Set CHECKPOINT_PATH_OVERRIDE secara eksplisit:\n"
            + "\n".join(str(p) for p in direct_matches)
        )

    # Fallback aman: hanya attention_only/seed_X/best.pt yang path parent-nya
    # mengandung AttentionOnly.
    recursive_matches = []

    for candidate in MODEL_ROOT.glob(
        f"**/{VARIANT}/seed_{seed}/best.pt"
    ):
        normalized = str(candidate).lower().replace("_", "")
        if "attentiononly" in normalized:
            recursive_matches.append(candidate)

    recursive_matches = sorted(
        dict.fromkeys(recursive_matches)
    )

    if len(recursive_matches) == 1:
        return recursive_matches[0]

    if len(recursive_matches) == 0:
        raise FileNotFoundError(
            f"best.pt Attention-Only seed {seed} tidak ditemukan di {MODEL_ROOT}.\n"
            "Pastikan checkpoint training sudah tersimpan atau isi "
            "CHECKPOINT_PATH_OVERRIDE."
        )

    raise RuntimeError(
        "Ditemukan beberapa best.pt Attention-Only untuk seed yang sama. "
        "Agar tidak salah checkpoint, set CHECKPOINT_PATH_OVERRIDE:\n"
        + "\n".join(str(p) for p in recursive_matches)
    )

CHECKPOINT_PATH = resolve_checkpoint(RUN_SEED)

EVAL_OUTPUT_DIR = (
    MODEL_ROOT
    / "HeldOut_Evaluation"
    / "AttentionOnly"
    / f"seed_{RUN_SEED}"
)

# ============================================================
# HELD-OUT DATASET
# ============================================================
BENCHMARK_ROOT = Path(
    "/content/drive/MyDrive/IVS/ViolenceDataset/Benchmark"
)

# RWF memakai cache lama yang sama dengan eksperimen sebelumnya jika tersedia.
EXISTING_CACHE_DIR = MODEL_ROOT / "new_saved_dataset"

# UCF dan Bus menggunakan shared held-out cache yang sama untuk semua model/seed.
# Dengan demikian paired comparison/McNemar nanti memakai sample yang identik.
HELDOUT_CACHE_DIR = MODEL_ROOT / "heldout_cache_rgb64_v2"

HELDOUT_SPECS = {
    "RWF": {
        "raw_dir": BENCHMARK_ROOT / "RWF",
        "cache_name": "RWF",
        "cache_dir": EXISTING_CACHE_DIR,
        "nonviolence_dir": None,
        "violence_dir": None,
    },
    "UCF": {
        "raw_dir": BENCHMARK_ROOT / "UCF",
        "cache_name": "UCF",
        "cache_dir": HELDOUT_CACHE_DIR,
        "nonviolence_dir": None,
        "violence_dir": None,
    },
    "BV": {
        "raw_dir": BENCHMARK_ROOT / "BV",
        "cache_name": "BV",
        "cache_dir": HELDOUT_CACHE_DIR,
        "nonviolence_dir": None,
        "violence_dir": None,
    },
}

# Auto-detection folder kelas. Jika dataset Anda memakai nama lain,
# isi nonviolence_dir / violence_dir pada HELDOUT_SPECS.
NONVIOLENCE_ALIASES = (
    "NonViolence",
    "Non-Violence",
    "Non_Violence",
    "Nonviolent",
    "NonViolent",
    "NonFight",
    "Non-Fight",
    "Non_Fight",
    "Normal",
    "NormalVideos",
    "Normal_Videos",
)

VIOLENCE_ALIASES = (
    "Violence",
    "Violent",
    "Fight",
    "Fighting",
    "FightVideos",
    "ViolentVideos",
    "Abnormal",
)

# ============================================================
# PREPROCESSING — SAMA DENGAN TRAINING FRAMEMEAN
# ============================================================
CLASS_NAMES = ["NonViolence", "Violence"]
NUM_CLASSES = 2

SEQUENCE_LENGTH = 16
RAW_HEIGHT = 64
RAW_WIDTH = 64
BACKBONE_INPUT_SIZE = 224
LSTM_UNITS = 32
BIDIRECTIONAL = True

PREFER_LEGACY_CACHE = True
LEGACY_CACHE_COLOR_ORDER = "BGR"
AUTO_SCALE_LEGACY_CACHE = True

# Evaluation
EVAL_BATCH_SIZE = 4   # sama dengan micro-batch training; aman untuk T=16
NUM_WORKERS = 2
USE_AMP = False       # training reference memakai FP32
BOOTSTRAP_SAMPLES = 1000
BOOTSTRAP_SEED = 2026

# Jika cache UCF/Bus belum ada, izinkan dibuat dari raw clips.
ALLOW_BUILD_CACHE = True

EVAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
HELDOUT_CACHE_DIR.mkdir(parents=True, exist_ok=True)

print("RUN_SEED        :", RUN_SEED)
print("CHECKPOINT_PATH :", CHECKPOINT_PATH)
print("OUTPUT          :", EVAL_OUTPUT_DIR)

RUN_SEED        : 42
CHECKPOINT_PATH : /content/drive/MyDrive/IVS/ViolenceDetection/Models/EffTempNet/EffTempNet_AttentionOnly_NoLS_Seed42_v1/attention_only/seed_42/best.pt
OUTPUT          : /content/drive/MyDrive/IVS/ViolenceDetection/Models/EffTempNet/HeldOut_Evaluation/AttentionOnly/seed_42


In [5]:
# ============================================================
# REPRODUCIBILITY / SAFETY
# ============================================================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_ENABLED = bool(USE_AMP and DEVICE.type == "cuda")
AMP_DTYPE = (
    torch.bfloat16
    if AMP_ENABLED and torch.cuda.is_bf16_supported()
    else torch.float16
)

def amp_context():
    if not AMP_ENABLED:
        return nullcontext()
    return torch.autocast(
        device_type="cuda",
        dtype=AMP_DTYPE,
        enabled=True,
    )

def set_global_seed(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def seed_worker(worker_id: int) -> None:
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)

def assert_finite_tensor(name: str, tensor: torch.Tensor) -> None:
    if not torch.isfinite(tensor).all():
        nan_count = int(torch.isnan(tensor).sum().item())
        inf_count = int(torch.isinf(tensor).sum().item())
        raise FloatingPointError(
            f"{name} mengandung NaN/Inf: nan={nan_count}, inf={inf_count}"
        )

def safe_torch_load(path: Path, map_location):
    try:
        return torch.load(
            path,
            map_location=map_location,
            weights_only=False,
        )
    except TypeError:
        return torch.load(
            path,
            map_location=map_location,
        )

set_global_seed(BOOTSTRAP_SEED)
print("Device:", DEVICE, "| AMP:", AMP_ENABLED)

Device: cuda | AMP: False


## 2. Model Attention-Only — identik dengan training

Checkpoint dievaluasi dengan arsitektur berikut:

- backbone `efficientformerv2_s0`, `num_classes=0`;
- feature dropout = 0.2;
- BiLSTM `hidden_size=32`, `batch_first=True`, `bidirectional=True`;
- `TemporalAttention`:
  - `Linear(64,64)`;
  - learnable bias berukuran 64;
  - `tanh`;
  - `Linear(64,1)`;
  - softmax sepanjang dimensi waktu;
  - weighted sum output BiLSTM;
- classifier `64 → 256 → 128 → 64 → 32 → 2`;
- **tidak ada** temporal Sum/Mean/Max residual branch.

Untuk evaluasi, model dibuat dengan `pretrained=False` karena seluruh bobot akan
dioverwrite oleh checkpoint. `load_state_dict(..., strict=True)` digunakan untuk
mencegah checkpoint yang tidak kompatibel dipakai secara diam-diam.

In [6]:
class TemporalAttention(nn.Module):
    def __init__(
        self,
        input_dim: int,
    ):
        super().__init__()

        self.projection = nn.Linear(
            input_dim,
            input_dim,
        )

        self.bias = nn.Parameter(
            torch.zeros(input_dim)
        )

        self.score = nn.Linear(
            input_dim,
            1,
        )

    def forward(
        self,
        sequence: torch.Tensor,
    ):
        if sequence.ndim != 3:
            raise ValueError(
                f"TemporalAttention expects [B,T,D], got {tuple(sequence.shape)}"
            )

        hidden = torch.tanh(
            self.projection(sequence)
            + self.bias
        )

        scores = self.score(
            hidden
        )

        weights = torch.softmax(
            scores,
            dim=1,
        )

        weighted_context = torch.sum(
            sequence * weights,
            dim=1,
        )

        return (
            weighted_context,
            weights.squeeze(-1),
        )


class EffTempNetAttentionOnly(nn.Module):
    VALID_VARIANTS = {
        "attention_only"
    }

    def __init__(
        self,
        variant: str = "attention_only",
        backbone_name: str = "efficientformerv2_s0",
        num_classes: int = NUM_CLASSES,
        lstm_units: int = LSTM_UNITS,
        pretrained: bool = False,
    ):
        super().__init__()

        if variant not in self.VALID_VARIANTS:
            raise ValueError(
                f"Unknown variant: {variant}"
            )

        self.variant = variant

        self.backbone = create_model(
            backbone_name,
            pretrained=pretrained,
            num_classes=0,
            in_chans=3,
        )

        self.feature_dim = int(
            self.backbone.num_features
        )

        self.feature_dropout = nn.Dropout(
            0.2
        )

        self.lstm = nn.LSTM(
            input_size=self.feature_dim,
            hidden_size=lstm_units,
            batch_first=True,
            bidirectional=True,
        )

        self.temporal_dim = 2 * lstm_units

        # Classifier identik dengan training Attention-Only / TempSum head.
        self.classifier = nn.Sequential(
            nn.Linear(self.temporal_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, num_classes),
        )

        self._initialize_classifier()

        # Training notebook membuat attention SETELAH classifier.
        self.attention = TemporalAttention(
            self.temporal_dim
        )

    def _initialize_classifier(
        self,
    ):
        for module in self.classifier.modules():
            if isinstance(module, nn.Linear):
                nn.init.kaiming_uniform_(
                    module.weight,
                    nonlinearity="relu",
                )

                if module.bias is not None:
                    nn.init.zeros_(
                        module.bias
                    )

    def forward(
        self,
        clip: torch.Tensor,
        return_attention: bool = False,
    ):
        if clip.ndim != 5:
            raise ValueError(
                f"Expected [B,T,C,H,W], got {tuple(clip.shape)}"
            )

        (
            batch_size,
            seq_len,
            channels,
            height,
            width,
        ) = clip.shape

        if seq_len != SEQUENCE_LENGTH:
            raise ValueError(
                f"Attention-Only expects T={SEQUENCE_LENGTH}, got T={seq_len}"
            )

        if channels != 3:
            raise ValueError(
                f"Expected 3 channels, got {channels}"
            )

        x = clip.reshape(
            batch_size * seq_len,
            channels,
            height,
            width,
        )

        x = F.interpolate(
            x,
            size=(
                BACKBONE_INPUT_SIZE,
                BACKBONE_INPUT_SIZE,
            ),
            mode="bilinear",
            align_corners=False,
        )

        features = self.backbone(
            x
        )

        if features.ndim == 4:
            features = F.adaptive_avg_pool2d(
                features,
                output_size=1,
            ).flatten(1)

        elif features.ndim != 2:
            features = features.flatten(
                1
            )

        features = features.view(
            batch_size,
            seq_len,
            -1,
        )

        features = self.feature_dropout(
            features
        )

        temporal_features, _ = self.lstm(
            features
        )

        # ATTENTION ONLY:
        # Tidak ada residual/sum/mean/max branch.
        context, attention_weights = self.attention(
            temporal_features
        )

        logits = self.classifier(
            context
        )

        if return_attention:
            return (
                logits,
                attention_weights,
            )

        return logits

## 3. Resolve / build held-out cache

Format cache dibuat sama dengan training FrameMean:

```text
[N, 16, 64, 64, 3]
```

Untuk raw video:
- 16 posisi diambil dengan `np.linspace(0, N-1, 16)`;
- OpenCV membaca frame;
- BGR → RGB;
- resize 64×64;
- skala `[0,1]`;
- jika beberapa posisi terakhir gagal dibaca, frame valid terakhir diulang.

Cache UCF dan Bus disimpan bersama dan dipakai ulang pada seed/model lain.

In [7]:
VIDEO_EXTENSIONS = {
    ".mp4", ".avi", ".mov", ".mkv",
    ".mpeg", ".mpg", ".webm"
}

def direct_subdirs(root: Path) -> Dict[str, Path]:
    if not root.exists():
        return {}
    return {
        p.name: p
        for p in root.iterdir()
        if p.is_dir()
    }

def resolve_one_class_dir(
    root: Path,
    explicit_name: Optional[str],
    aliases: Tuple[str, ...],
    canonical_name: str,
) -> Path:
    if explicit_name is not None:
        candidate = root / explicit_name
        if not candidate.exists():
            raise FileNotFoundError(
                f"Folder {canonical_name} tidak ditemukan: {candidate}"
            )
        return candidate

    subdirs = direct_subdirs(root)
    lower_map = {
        name.lower(): path
        for name, path in subdirs.items()
    }

    matches = []
    for alias in aliases:
        candidate = lower_map.get(alias.lower())
        if candidate is not None and candidate not in matches:
            matches.append(candidate)

    if len(matches) != 1:
        raise RuntimeError(
            f"Tidak dapat menentukan folder kelas {canonical_name} "
            f"secara aman pada {root}.\n"
            f"Matches: {[p.name for p in matches]}\n"
            f"Available: {sorted(subdirs.keys())}\n"
            "Isi nonviolence_dir/violence_dir di HELDOUT_SPECS."
        )

    return matches[0]

def list_videos_for_spec(
    raw_dir: Path,
    nonviolence_dir: Optional[str],
    violence_dir: Optional[str],
) -> List[Tuple[Path, int]]:
    if not raw_dir.exists():
        raise FileNotFoundError(
            f"Raw held-out directory tidak ditemukan: {raw_dir}"
        )

    class_dirs = {
        0: resolve_one_class_dir(
            raw_dir,
            nonviolence_dir,
            NONVIOLENCE_ALIASES,
            "NonViolence",
        ),
        1: resolve_one_class_dir(
            raw_dir,
            violence_dir,
            VIOLENCE_ALIASES,
            "Violence",
        ),
    }

    records = []

    for label, class_dir in class_dirs.items():
        videos = sorted(
            p for p in class_dir.rglob("*")
            if p.is_file() and p.suffix.lower() in VIDEO_EXTENSIONS
        )

        if not videos:
            raise RuntimeError(
                f"Tidak ada video pada folder kelas: {class_dir}"
            )

        print(
            f"{CLASS_NAMES[label]:12s}: {len(videos):5d} clips | {class_dir}"
        )

        records.extend(
            (p, label)
            for p in videos
        )

    labels = np.asarray(
        [label for _, label in records],
        dtype=np.int64,
    )

    if set(np.unique(labels).tolist()) != {0, 1}:
        raise RuntimeError(
            "Held-out dataset harus mempunyai kedua kelas: "
            "NonViolence dan Violence."
        )

    return records

def extract_uniform_rgb_frames(
    video_path: Path,
    sequence_length: int = SEQUENCE_LENGTH,
    height: int = RAW_HEIGHT,
    width: int = RAW_WIDTH,
) -> Optional[np.ndarray]:
    cap = cv2.VideoCapture(str(video_path))

    if not cap.isOpened():
        return None

    frame_count = int(
        cap.get(cv2.CAP_PROP_FRAME_COUNT)
    )

    if frame_count <= 0:
        cap.release()
        return None

    frame_positions = np.linspace(
        0,
        max(frame_count - 1, 0),
        sequence_length,
    ).astype(int)

    frames = []

    for pos in frame_positions:
        cap.set(
            cv2.CAP_PROP_POS_FRAMES,
            int(pos),
        )

        ok, frame_bgr = cap.read()

        if not ok:
            break

        frame_rgb = cv2.cvtColor(
            frame_bgr,
            cv2.COLOR_BGR2RGB,
        )

        frame_rgb = cv2.resize(
            frame_rgb,
            (width, height),
            interpolation=cv2.INTER_LINEAR,
        )

        frames.append(
            frame_rgb.astype(np.float32) / 255.0
        )

    cap.release()

    if not frames:
        return None

    while len(frames) < sequence_length:
        frames.append(frames[-1].copy())

    return np.stack(
        frames[:sequence_length],
        axis=0,
    ).astype(np.float32)

def build_rgb_cache(
    raw_dir: Path,
    cache_dir: Path,
    cache_name: str,
    nonviolence_dir: Optional[str],
    violence_dir: Optional[str],
):
    cache_dir.mkdir(parents=True, exist_ok=True)

    feature_path = cache_dir / f"{cache_name}_rgb64_v2_features.npy"
    label_path = cache_dir / f"{cache_name}_rgb64_v2_labels.npy"
    paths_path = cache_dir / f"{cache_name}_rgb64_v2_paths.json"

    records = list_videos_for_spec(
        raw_dir,
        nonviolence_dir,
        violence_dir,
    )

    # Gunakan temporary memmap agar tidak menyimpan semua clip di RAM.
    n_requested = len(records)

    temp_path = cache_dir / f".{cache_name}_tmp_features.npy"

    if temp_path.exists():
        temp_path.unlink()

    temp = np.lib.format.open_memmap(
        temp_path,
        mode="w+",
        dtype=np.float32,
        shape=(
            n_requested,
            SEQUENCE_LENGTH,
            RAW_HEIGHT,
            RAW_WIDTH,
            3,
        ),
    )

    labels_buffer = np.empty(
        n_requested,
        dtype=np.int64,
    )

    valid_paths = []
    failed_paths = []
    valid_count = 0

    for video_path, label in tqdm(
        records,
        desc=f"Build cache {cache_name}",
        unit="video",
    ):
        clip = extract_uniform_rgb_frames(video_path)

        if clip is None:
            failed_paths.append(str(video_path))
            continue

        temp[valid_count] = clip
        labels_buffer[valid_count] = int(label)
        valid_paths.append(str(video_path))
        valid_count += 1

    temp.flush()
    del temp

    if valid_count == 0:
        if temp_path.exists():
            temp_path.unlink()
        raise RuntimeError(
            f"Semua video gagal didekode untuk {cache_name}."
        )

    # Compact hanya sample valid ke final .npy.
    temp_read = np.load(
        temp_path,
        mmap_mode="r",
    )

    final = np.lib.format.open_memmap(
        feature_path,
        mode="w+",
        dtype=np.float32,
        shape=(
            valid_count,
            SEQUENCE_LENGTH,
            RAW_HEIGHT,
            RAW_WIDTH,
            3,
        ),
    )

    chunk_size = 16

    for start in range(0, valid_count, chunk_size):
        end = min(start + chunk_size, valid_count)
        final[start:end] = temp_read[start:end]

    final.flush()
    del final, temp_read

    np.save(
        label_path,
        labels_buffer[:valid_count],
    )

    with open(
        paths_path,
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            {
                "paths": valid_paths,
                "failed": failed_paths,
                "n_requested": n_requested,
                "n_valid": valid_count,
                "color_order": "RGB",
                "sequence_length": SEQUENCE_LENGTH,
                "height": RAW_HEIGHT,
                "width": RAW_WIDTH,
            },
            f,
            indent=2,
        )

    if temp_path.exists():
        temp_path.unlink()

    print(
        f"Saved {feature_path} | "
        f"shape=({valid_count},{SEQUENCE_LENGTH},{RAW_HEIGHT},{RAW_WIDTH},3)"
    )
    print("Failed videos:", len(failed_paths))

    return {
        "feature_path": feature_path,
        "label_path": label_path,
        "paths_path": paths_path,
        "color_order": "RGB",
        "cache_source": "built_rgb64_v2",
    }

def resolve_heldout_store(
    dataset_name: str,
    spec: Dict,
) -> Dict:
    raw_dir = Path(spec["raw_dir"])
    cache_dir = Path(spec["cache_dir"])
    cache_name = str(spec["cache_name"])

    rgb_feature = cache_dir / f"{cache_name}_rgb64_v2_features.npy"
    rgb_label = cache_dir / f"{cache_name}_rgb64_v2_labels.npy"
    rgb_paths = cache_dir / f"{cache_name}_rgb64_v2_paths.json"

    legacy_feature = cache_dir / f"{cache_name}_features.npy"
    legacy_label = cache_dir / f"{cache_name}_labels.npy"
    legacy_paths = cache_dir / f"{cache_name}_paths.json"

    # Sama dengan training: RGB-v2 diprioritaskan; legacy dipakai jika v2 tidak ada.
    if rgb_feature.exists() and rgb_label.exists():
        print(f"✅ {dataset_name}: RGB-v2 cache")
        return {
            "feature_path": rgb_feature,
            "label_path": rgb_label,
            "paths_path": rgb_paths if rgb_paths.exists() else None,
            "color_order": "RGB",
            "cache_source": "rgb64_v2",
        }

    if (
        PREFER_LEGACY_CACHE
        and legacy_feature.exists()
        and legacy_label.exists()
    ):
        print(
            f"✅ {dataset_name}: legacy cache "
            f"({LEGACY_CACHE_COLOR_ORDER}→RGB saat loading)"
        )
        return {
            "feature_path": legacy_feature,
            "label_path": legacy_label,
            "paths_path": legacy_paths if legacy_paths.exists() else None,
            "color_order": LEGACY_CACHE_COLOR_ORDER,
            "cache_source": "legacy",
        }

    if not ALLOW_BUILD_CACHE:
        raise FileNotFoundError(
            f"Cache {dataset_name} tidak ada dan ALLOW_BUILD_CACHE=False."
        )

    if not raw_dir.exists():
        raise FileNotFoundError(
            f"Cache dan raw folder tidak ditemukan untuk {dataset_name}.\n"
            f"Raw expected: {raw_dir}"
        )

    print(
        f"ℹ️ {dataset_name}: cache belum ada, dibuat dari {raw_dir}"
    )

    return build_rgb_cache(
        raw_dir=raw_dir,
        cache_dir=cache_dir,
        cache_name=cache_name,
        nonviolence_dir=spec.get("nonviolence_dir"),
        violence_dir=spec.get("violence_dir"),
    )

In [8]:
IMAGENET_MEAN = torch.tensor(
    [0.485, 0.456, 0.406],
    dtype=torch.float32,
).view(1, 3, 1, 1)

IMAGENET_STD = torch.tensor(
    [0.229, 0.224, 0.225],
    dtype=torch.float32,
).view(1, 3, 1, 1)

class HeldOutFrameMeanDataset(Dataset):
    def __init__(
        self,
        feature_path: Path,
        label_path: Path,
        color_order: str,
    ):
        self.feature_path = Path(feature_path)
        self.label_path = Path(label_path)

        self.features = np.load(
            self.feature_path,
            mmap_mode="r",
        )

        self.labels = np.load(
            self.label_path,
            mmap_mode="r",
        ).astype(np.int64)

        self.color_order = str(color_order).upper()

        if len(self.features) != len(self.labels):
            raise ValueError(
                f"Feature/label mismatch: "
                f"{len(self.features)} vs {len(self.labels)}"
            )

        expected_shape = (
            SEQUENCE_LENGTH,
            RAW_HEIGHT,
            RAW_WIDTH,
            3,
        )

        if (
            self.features.ndim != 5
            or tuple(self.features.shape[1:]) != expected_shape
        ):
            raise ValueError(
                f"Unexpected cache shape {self.features.shape}; "
                f"expected [N,{SEQUENCE_LENGTH},"
                f"{RAW_HEIGHT},{RAW_WIDTH},3]."
            )

        unique_labels = set(
            np.unique(self.labels).tolist()
        )

        if unique_labels != {0, 1}:
            raise ValueError(
                "Held-out dataset harus mempunyai label {0,1}; "
                f"ditemukan {sorted(unique_labels)}."
            )

        if self.color_order not in {"RGB", "BGR"}:
            raise ValueError(
                f"Unsupported color_order={self.color_order}"
            )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index: int):
        frames = np.asarray(
            self.features[index],
            dtype=np.float32,
        )

        label = int(self.labels[index])

        if not np.isfinite(frames).all():
            raise ValueError(
                f"NaN/Inf pada cache sample index={index}"
            )

        frame_min = float(frames.min())
        frame_max = float(frames.max())

        if (
            AUTO_SCALE_LEGACY_CACHE
            and frame_min >= 0.0
            and frame_max > 1.5
        ):
            if frame_max <= 255.0 + 1e-3:
                frames = frames / 255.0
            else:
                raise ValueError(
                    f"Invalid cache range [{frame_min},{frame_max}] "
                    f"at sample={index}"
                )

        if (
            float(frames.min()) < -1e-4
            or float(frames.max()) > 1.0001
        ):
            raise ValueError(
                f"Cache harus [0,1], ditemukan "
                f"[{float(frames.min())},{float(frames.max())}] "
                f"at sample={index}"
            )

        if self.color_order == "BGR":
            frames = frames[..., [2, 1, 0]]

        clip = torch.from_numpy(
            np.ascontiguousarray(frames)
        ).permute(
            0,
            3,
            1,
            2,
        )

        clip = (
            clip
            - IMAGENET_MEAN
        ) / IMAGENET_STD

        return (
            clip,
            torch.tensor(label, dtype=torch.long),
            int(index),
        )

def load_paths_if_available(
    paths_path: Optional[Path],
    expected_n: int,
) -> Optional[List[str]]:
    if paths_path is None:
        return None

    path = Path(paths_path)

    if not path.exists():
        return None

    with open(
        path,
        "r",
        encoding="utf-8",
    ) as f:
        payload = json.load(f)

    paths = payload.get("paths")

    if not isinstance(paths, list):
        return None

    if len(paths) != expected_n:
        print(
            f"⚠️ paths.json count {len(paths)} != {expected_n}; "
            "video_path tidak dipakai."
        )
        return None

    return [str(p) for p in paths]

def audit_dataset(
    dataset_name: str,
    store: Dict,
) -> Dict:
    dataset = HeldOutFrameMeanDataset(
        store["feature_path"],
        store["label_path"],
        store["color_order"],
    )

    labels = np.asarray(
        dataset.labels,
        dtype=np.int64,
    )

    counts = np.bincount(
        labels,
        minlength=2,
    )

    audit_indices = np.unique(
        np.linspace(
            0,
            len(dataset) - 1,
            min(6, len(dataset)),
        ).astype(int)
    )

    for idx in audit_indices:
        clip, label, sample_idx = dataset[int(idx)]

        if tuple(clip.shape) != (
            SEQUENCE_LENGTH,
            3,
            RAW_HEIGHT,
            RAW_WIDTH,
        ):
            raise RuntimeError(
                f"Invalid clip shape at {sample_idx}: {tuple(clip.shape)}"
            )

        assert_finite_tensor(
            f"{dataset_name}_sample_{sample_idx}",
            clip,
        )

    print(
        f"✅ {dataset_name}: N={len(dataset)}, "
        f"NonViolence={counts[0]}, Violence={counts[1]}, "
        f"cache={store['cache_source']}"
    )

    return {
        "dataset": dataset_name,
        "n_total": int(len(dataset)),
        "n_nonviolence": int(counts[0]),
        "n_violence": int(counts[1]),
        "feature_path": str(store["feature_path"]),
        "label_path": str(store["label_path"]),
        "color_order_in_cache": store["color_order"],
        "cache_source": store["cache_source"],
    }

def make_loader(dataset: Dataset):
    generator = torch.Generator()
    generator.manual_seed(BOOTSTRAP_SEED)

    return DataLoader(
        dataset,
        batch_size=EVAL_BATCH_SIZE,
        shuffle=False,
        drop_last=False,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE.type == "cuda"),
        worker_init_fn=seed_worker,
        generator=generator,
        persistent_workers=(NUM_WORKERS > 0),
    )

## 4. Load dan audit checkpoint `best.pt`

Evaluasi **selalu menggunakan `best.pt`**, bukan `last.pt`.

Checkpoint harus berasal dari:

```python
variant = "attention_only"
```

pada seed yang dipilih.

Audit akan memeriksa:
- checkpoint seed bila metadata tersedia;
- `strict=True` state-dict loading;
- BiLSTM hidden size = 32;
- bidirectional = True;
- modul attention aktif;
- attention weights berukuran `[B,16]`;
- setiap baris attention weights menjumlah ≈ 1;
- output logits berukuran `[B,2]`.

In [9]:
if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(
        f"Checkpoint tidak ditemukan: {CHECKPOINT_PATH}"
    )

checkpoint = safe_torch_load(
    CHECKPOINT_PATH,
    map_location="cpu",
)

if not isinstance(checkpoint, dict):
    raise TypeError(
        "Checkpoint bukan dictionary dari training notebook."
    )

if "model_state_dict" not in checkpoint:
    raise KeyError(
        "Checkpoint tidak mempunyai key 'model_state_dict'."
    )

checkpoint_config = checkpoint.get("config", {})

expected_config = {
    "variant": VARIANT,
    "seed": RUN_SEED,
    "sequence_length": SEQUENCE_LENGTH,
    "backbone_input_size": BACKBONE_INPUT_SIZE,
}

mismatches = []

for key, expected in expected_config.items():
    if key in checkpoint_config:
        actual = checkpoint_config[key]
        if actual != expected:
            mismatches.append(
                (key, actual, expected)
            )

if mismatches:
    raise RuntimeError(
        "Checkpoint config mismatch:\n"
        + "\n".join(
            f"- {key}: actual={actual}, expected={expected}"
            for key, actual, expected in mismatches
        )
    )

model = EffTempNetAttentionOnly(
    variant=VARIANT,
    lstm_units=LSTM_UNITS,
    pretrained=False,
).to(DEVICE)

load_result = model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True,
)

model.eval()

assert model.lstm is not None
assert model.lstm.bidirectional is True
assert model.lstm.hidden_size == LSTM_UNITS
assert model.attention is not None
assert isinstance(model.attention, TemporalAttention)

# Smoke forward dengan shape aktual FrameMean.
with torch.inference_mode():
    dummy = torch.zeros(
        1,
        SEQUENCE_LENGTH,
        3,
        RAW_HEIGHT,
        RAW_WIDTH,
        device=DEVICE,
    )

    dummy_logits, dummy_attention = model(
        dummy,
        return_attention=True,
    )

if tuple(dummy_logits.shape) != (1, NUM_CLASSES):
    raise RuntimeError(
        f"Unexpected output shape: {tuple(dummy_logits.shape)}"
    )

if tuple(dummy_attention.shape) != (1, SEQUENCE_LENGTH):
    raise RuntimeError(
        f"Unexpected attention shape: {tuple(dummy_attention.shape)}"
    )

attention_row_sum = dummy_attention.sum(dim=1)

if not torch.allclose(
    attention_row_sum,
    torch.ones_like(attention_row_sum),
    atol=1e-5,
    rtol=1e-5,
):
    raise RuntimeError(
        f"Attention weights do not sum to 1: {attention_row_sum}"
    )

if (dummy_attention < 0).any() or (dummy_attention > 1).any():
    raise RuntimeError(
        "Attention weights outside [0,1]."
    )

print("✅ Attention-Only checkpoint audit PASS")
print("Checkpoint :", CHECKPOINT_PATH)
print("Epoch      :", checkpoint.get("epoch", "unknown"))
print("Best val F1:", checkpoint.get("best_score", "unknown"))
print("Load result:", load_result)
print("Feature dim  :", model.feature_dim)
print("LSTM hidden  :", model.lstm.hidden_size)
print("Bidirectional:", model.lstm.bidirectional)
print("Temporal dim :", model.temporal_dim)
print("Attention    :", type(model.attention).__name__)
print(
    "Total parameters:",
    sum(p.numel() for p in model.parameters())
)

del dummy, dummy_logits, dummy_attention

if torch.cuda.is_available():
    torch.cuda.empty_cache()

✅ Attention-Only checkpoint audit PASS
Checkpoint : /content/drive/MyDrive/IVS/ViolenceDetection/Models/EffTempNet/EffTempNet_AttentionOnly_NoLS_Seed42_v1/attention_only/seed_42/best.pt
Epoch      : 45
Best val F1: 0.8884290690484791
Load result: <All keys matched successfully>
Feature dim  : 176
LSTM hidden  : 32
Bidirectional: True
Temporal dim : 64
Attention    : TemporalAttention
Total parameters: 3364243


## 5. Resolve dan audit tiga held-out dataset

In [10]:
heldout_stores = {}
dataset_audit_rows = []

for dataset_name, spec in HELDOUT_SPECS.items():
    print("\n" + "=" * 90)
    print("DATASET:", dataset_name)
    print("=" * 90)

    store = resolve_heldout_store(
        dataset_name,
        spec,
    )

    heldout_stores[dataset_name] = store

    dataset_audit_rows.append(
        audit_dataset(
            dataset_name,
            store,
        )
    )

dataset_audit_df = pd.DataFrame(dataset_audit_rows)

dataset_audit_path = (
    EVAL_OUTPUT_DIR
    / "heldout_dataset_audit.csv"
)

dataset_audit_df.to_csv(
    dataset_audit_path,
    index=False,
)

display(dataset_audit_df)

print("Saved:", dataset_audit_path)


DATASET: RWF
✅ RWF: legacy cache (BGR→RGB saat loading)
✅ RWF: N=1998, NonViolence=1000, Violence=998, cache=legacy

DATASET: UCF
✅ UCF: RGB-v2 cache
✅ UCF: N=100, NonViolence=50, Violence=50, cache=rgb64_v2

DATASET: BV
✅ BV: RGB-v2 cache
✅ BV: N=1400, NonViolence=700, Violence=700, cache=rgb64_v2


,dataset,n_total,n_nonviolence,n_violence,feature_path,label_path,color_order_in_cache,cache_source
0,RWF,1998,1000,998,/content/drive/MyDrive/IVS/ViolenceDetection/M...,/content/drive/MyDrive/IVS/ViolenceDetection/M...,BGR,legacy
1,UCF,100,50,50,/content/drive/MyDrive/IVS/ViolenceDetection/M...,/content/drive/MyDrive/IVS/ViolenceDetection/M...,RGB,rgb64_v2
2,BV,1400,700,700,/content/drive/MyDrive/IVS/ViolenceDetection/M...,/content/drive/MyDrive/IVS/ViolenceDetection/M...,RGB,rgb64_v2


Saved: /content/drive/MyDrive/IVS/ViolenceDetection/Models/EffTempNet/HeldOut_Evaluation/AttentionOnly/seed_42/heldout_dataset_audit.csv


## 6. Evaluation metrics, bootstrap CI, prediction export, dan attention weights

`predictions.csv` menyimpan:

- `sample_index`
- `sample_key`
- `video_path` jika tersedia
- `y_true`
- `y_pred`
- `is_correct`
- `p_nonviolence`
- `p_violence`
- `attention_t01` ... `attention_t16`

Kolom prediction utama identik dengan evaluator model lain sehingga bisa
dipasangkan clip-by-clip untuk McNemar test dan paired bootstrap
\(\Delta AP\).

Attention weights hanya merupakan output diagnostik model. Tidak digunakan
untuk memilih checkpoint atau menyesuaikan model pada held-out dataset.

In [11]:
def compute_classification_metrics(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    p_violence: np.ndarray,
) -> Dict[str, float]:
    precision, recall, f1, support = (
        precision_recall_fscore_support(
            y_true,
            y_pred,
            labels=[0, 1],
            zero_division=0,
        )
    )

    result = {
        "n": int(len(y_true)),
        "accuracy": float(
            accuracy_score(y_true, y_pred)
        ),
        "balanced_accuracy": float(
            balanced_accuracy_score(y_true, y_pred)
        ),
        "precision_nonviolence": float(precision[0]),
        "recall_nonviolence": float(recall[0]),
        "f1_nonviolence": float(f1[0]),
        "precision_violence": float(precision[1]),
        "recall_violence": float(recall[1]),
        "f1_violence": float(f1[1]),
        "f1_macro": float(
            f1_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0,
            )
        ),
    }

    if len(np.unique(y_true)) == 2:
        result["roc_auc"] = float(
            roc_auc_score(y_true, p_violence)
        )

        result["average_precision"] = float(
            average_precision_score(
                y_true,
                p_violence,
            )
        )
    else:
        result["roc_auc"] = float("nan")
        result["average_precision"] = float("nan")

    return result

@torch.inference_mode()
def evaluate_loader(
    model: nn.Module,
    loader: DataLoader,
    device: torch.device,
):
    model.eval()

    all_indices = []
    all_labels = []
    all_predictions = []
    all_probabilities = []
    all_attention_weights = []

    for clips, labels, indices in tqdm(
        loader,
        desc="Inference",
        leave=False,
    ):
        clips = clips.to(
            device,
            non_blocking=True,
        )

        labels = labels.to(
            device,
            non_blocking=True,
        )

        assert_finite_tensor(
            "evaluation_input",
            clips,
        )

        with amp_context():
            logits, attention_weights = model(
                clips,
                return_attention=True,
            )

        assert_finite_tensor(
            "evaluation_logits",
            logits,
        )

        assert_finite_tensor(
            "evaluation_attention",
            attention_weights,
        )

        if attention_weights.ndim != 2:
            raise RuntimeError(
                f"Expected attention [B,T], got {tuple(attention_weights.shape)}"
            )

        if attention_weights.shape[1] != SEQUENCE_LENGTH:
            raise RuntimeError(
                f"Expected T={SEQUENCE_LENGTH} attention positions, "
                f"got {attention_weights.shape[1]}"
            )

        row_sums = attention_weights.float().sum(
            dim=1
        )

        if not torch.allclose(
            row_sums,
            torch.ones_like(row_sums),
            atol=1e-4,
            rtol=1e-4,
        ):
            raise RuntimeError(
                "Attention weights do not sum to one."
            )

        probabilities = torch.softmax(
            logits.float(),
            dim=1,
        )

        assert_finite_tensor(
            "evaluation_probabilities",
            probabilities,
        )

        predictions = torch.argmax(
            probabilities,
            dim=1,
        )

        all_indices.extend(
            indices.cpu().numpy().tolist()
        )

        all_labels.extend(
            labels.cpu().numpy().tolist()
        )

        all_predictions.extend(
            predictions.cpu().numpy().tolist()
        )

        all_probabilities.extend(
            probabilities.cpu().numpy().tolist()
        )

        all_attention_weights.extend(
            attention_weights.float().cpu().numpy().tolist()
        )

    sample_indices = np.asarray(
        all_indices,
        dtype=np.int64,
    )

    y_true = np.asarray(
        all_labels,
        dtype=np.int64,
    )

    y_pred = np.asarray(
        all_predictions,
        dtype=np.int64,
    )

    probs = np.asarray(
        all_probabilities,
        dtype=np.float64,
    )

    attention = np.asarray(
        all_attention_weights,
        dtype=np.float64,
    )

    if len(y_true) == 0:
        raise RuntimeError(
            "Evaluation menghasilkan 0 predictions."
        )

    if probs.shape != (
        len(y_true),
        NUM_CLASSES,
    ):
        raise RuntimeError(
            f"Invalid probability shape {probs.shape}"
        )

    if attention.shape != (
        len(y_true),
        SEQUENCE_LENGTH,
    ):
        raise RuntimeError(
            f"Invalid attention shape {attention.shape}"
        )

    metrics = compute_classification_metrics(
        y_true,
        y_pred,
        probs[:, 1],
    )

    return (
        metrics,
        sample_indices,
        y_true,
        y_pred,
        probs,
        attention,
    )

def bootstrap_ci(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    p_violence: np.ndarray,
    metric_name: str,
    n_bootstrap: int,
    seed: int,
    alpha: float = 0.05,
) -> Tuple[float, float]:
    rng = np.random.default_rng(seed)

    n = len(y_true)
    values = []

    attempts = 0
    max_attempts = max(
        n_bootstrap * 5,
        1000,
    )

    while (
        len(values) < n_bootstrap
        and attempts < max_attempts
    ):
        attempts += 1

        idx = rng.integers(
            0,
            n,
            size=n,
        )

        yt = y_true[idx]
        yp = y_pred[idx]
        pp = p_violence[idx]

        if (
            metric_name in {"roc_auc", "average_precision"}
            and len(np.unique(yt)) < 2
        ):
            continue

        if metric_name == "accuracy":
            value = accuracy_score(yt, yp)
        elif metric_name == "f1_macro":
            value = f1_score(
                yt,
                yp,
                average="macro",
                zero_division=0,
            )
        elif metric_name == "f1_violence":
            value = f1_score(
                yt,
                yp,
                pos_label=1,
                zero_division=0,
            )
        elif metric_name == "roc_auc":
            value = roc_auc_score(yt, pp)
        elif metric_name == "average_precision":
            value = average_precision_score(
                yt,
                pp,
            )
        else:
            raise ValueError(metric_name)

        values.append(float(value))

    if not values:
        return float("nan"), float("nan")

    return (
        float(np.quantile(values, alpha / 2)),
        float(np.quantile(values, 1 - alpha / 2)),
    )

def save_plots(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    p_violence: np.ndarray,
    title_prefix: str,
    output_dir: Path,
):
    output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1],
    )

    fig, ax = plt.subplots(
        figsize=(5, 4)
    )

    image = ax.imshow(cm)

    for (row, col), value in np.ndenumerate(cm):
        ax.text(
            col,
            row,
            str(value),
            ha="center",
            va="center",
        )

    ax.set_xticks(
        [0, 1],
        CLASS_NAMES,
    )
    ax.set_yticks(
        [0, 1],
        CLASS_NAMES,
    )
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(
        f"{title_prefix} — Confusion Matrix"
    )
    fig.colorbar(image, ax=ax)
    fig.tight_layout()
    fig.savefig(
        output_dir / "confusion_matrix.png",
        dpi=180,
    )
    plt.close(fig)

    if len(np.unique(y_true)) == 2:
        fpr, tpr, _ = roc_curve(
            y_true,
            p_violence,
        )

        auc_value = roc_auc_score(
            y_true,
            p_violence,
        )

        fig, ax = plt.subplots(
            figsize=(5, 4)
        )
        ax.plot(
            fpr,
            tpr,
            label=f"AUC={auc_value:.4f}",
        )
        ax.plot(
            [0, 1],
            [0, 1],
            linestyle="--",
        )
        ax.set_xlabel(
            "False Positive Rate"
        )
        ax.set_ylabel(
            "True Positive Rate"
        )
        ax.set_title(
            f"{title_prefix} — ROC"
        )
        ax.legend()
        ax.grid(True)
        fig.tight_layout()
        fig.savefig(
            output_dir / "roc_curve.png",
            dpi=180,
        )
        plt.close(fig)

        pr_precision, pr_recall, _ = (
            precision_recall_curve(
                y_true,
                p_violence,
            )
        )

        ap_value = average_precision_score(
            y_true,
            p_violence,
        )

        fig, ax = plt.subplots(
            figsize=(5, 4)
        )
        ax.plot(
            pr_recall,
            pr_precision,
            label=f"AP={ap_value:.4f}",
        )
        ax.set_xlabel("Recall")
        ax.set_ylabel("Precision")
        ax.set_title(
            f"{title_prefix} — Precision-Recall"
        )
        ax.legend()
        ax.grid(True)
        fig.tight_layout()
        fig.savefig(
            output_dir / "precision_recall_curve.png",
            dpi=180,
        )
        plt.close(fig)

## 7. Jalankan held-out evaluation

Cell ini hanya melakukan forward inference pada checkpoint Attention-Only yang
sudah dibekukan. Tidak ada pembaruan bobot.

In [12]:
results = []

ci_metric_names = [
    "accuracy",
    "f1_macro",
    "f1_violence",
    "roc_auc",
    "average_precision",
]

for dataset_position, (
    dataset_name,
    store,
) in enumerate(
    heldout_stores.items()
):
    print("\n" + "=" * 90)
    print(
        f"EVALUATE {dataset_name} | "
        f"Attention-Only seed={RUN_SEED}"
    )
    print("=" * 90)

    dataset = HeldOutFrameMeanDataset(
        store["feature_path"],
        store["label_path"],
        store["color_order"],
    )

    loader = make_loader(dataset)

    (
        metrics,
        sample_indices,
        y_true,
        y_pred,
        probs,
        attention_weights,
    ) = evaluate_loader(
        model,
        loader,
        DEVICE,
    )

    output_dir = (
        EVAL_OUTPUT_DIR
        / dataset_name
    )
    output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    source_paths = load_paths_if_available(
        store.get("paths_path"),
        expected_n=len(dataset),
    )

    if source_paths is None:
        sample_keys = [
            f"{dataset_name}::{int(index)}"
            for index in sample_indices
        ]
        video_paths = [
            ""
            for _ in sample_indices
        ]
    else:
        video_paths = [
            source_paths[int(index)]
            for index in sample_indices
        ]
        sample_keys = [
            Path(path).as_posix()
            for path in video_paths
        ]

    prediction_df = pd.DataFrame(
        {
            "dataset": dataset_name,
            "variant": VARIANT,
            "seed": RUN_SEED,
            "sample_index": sample_indices,
            "sample_key": sample_keys,
            "video_path": video_paths,
            "y_true": y_true,
            "y_pred": y_pred,
            "is_correct": (
                y_true == y_pred
            ).astype(np.int64),
            "p_nonviolence": probs[:, 0],
            "p_violence": probs[:, 1],
        }
    )

    for temporal_index in range(
        SEQUENCE_LENGTH
    ):
        prediction_df[
            f"attention_t{temporal_index + 1:02d}"
        ] = attention_weights[
            :,
            temporal_index,
        ]

    # Per-clip diagnostic: temporal position with highest learned weight.
    prediction_df[
        "attention_argmax_1based"
    ] = (
        np.argmax(
            attention_weights,
            axis=1,
        )
        + 1
    )

    # Audit unik agar paired comparison/McNemar aman.
    if prediction_df["sample_key"].duplicated().any():
        duplicated = prediction_df.loc[
            prediction_df["sample_key"].duplicated(),
            "sample_key",
        ].head(10).tolist()

        raise RuntimeError(
            f"Duplicate sample_key pada {dataset_name}: {duplicated}"
        )

    prediction_path = (
        output_dir
        / "predictions.csv"
    )

    prediction_df.to_csv(
        prediction_path,
        index=False,
    )

    attention_summary_df = pd.DataFrame(
        {
            "temporal_position_1based": np.arange(
                1,
                SEQUENCE_LENGTH + 1,
                dtype=np.int64,
            ),
            "attention_mean": attention_weights.mean(
                axis=0
            ),
            "attention_std": attention_weights.std(
                axis=0,
                ddof=0,
            ),
            "attention_median": np.median(
                attention_weights,
                axis=0,
            ),
        }
    )

    attention_summary_df.to_csv(
        output_dir
        / "attention_summary.csv",
        index=False,
    )

    for metric_position, metric_name in enumerate(
        ci_metric_names
    ):
        ci_seed = (
            BOOTSTRAP_SEED
            + dataset_position * 100
            + metric_position
        )

        low, high = bootstrap_ci(
            y_true=y_true,
            y_pred=y_pred,
            p_violence=probs[:, 1],
            metric_name=metric_name,
            n_bootstrap=BOOTSTRAP_SAMPLES,
            seed=ci_seed,
        )

        metrics[
            f"{metric_name}_ci95_low"
        ] = low

        metrics[
            f"{metric_name}_ci95_high"
        ] = high

    metrics.update(
        {
            "variant": VARIANT,
            "seed": RUN_SEED,
            "dataset": dataset_name,
            "checkpoint_path": str(
                CHECKPOINT_PATH
            ),
            "best_epoch": checkpoint.get(
                "epoch",
                np.nan,
            ),
            "best_val_f1_macro": checkpoint.get(
                "best_score",
                np.nan,
            ),
            "cache_source": store[
                "cache_source"
            ],
        }
    )

    results.append(metrics)

    save_plots(
        y_true=y_true,
        y_pred=y_pred,
        p_violence=probs[:, 1],
        title_prefix=(
            f"Attention-Only seed {RUN_SEED} — {dataset_name}"
        ),
        output_dir=output_dir,
    )

    print(
        f"N={metrics['n']} | "
        f"AP={metrics['average_precision']:.4f} | "
        f"AUC={metrics['roc_auc']:.4f} | "
        f"Macro-F1={metrics['f1_macro']:.4f}"
    )

    del dataset, loader, probs, attention_weights

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

heldout_metrics_df = pd.DataFrame(results)

metrics_output = (
    EVAL_OUTPUT_DIR
    / f"heldout_metrics_attention_only_seed_{RUN_SEED}.csv"
)

heldout_metrics_df.to_csv(
    metrics_output,
    index=False,
)

display_columns = [
    "dataset",
    "n",
    "accuracy",
    "balanced_accuracy",
    "f1_macro",
    "precision_violence",
    "recall_violence",
    "f1_violence",
    "roc_auc",
    "average_precision",
]

print("\n=== HELD-OUT RESULTS ===")
display(
    heldout_metrics_df[
        display_columns
    ].round(4)
)

print("Saved:", metrics_output)


EVALUATE RWF | Attention-Only seed=42


Inference:   0%|          | 0/500 [00:00<?, ?it/s]

N=1998 | AP=0.6165 | AUC=0.6231 | Macro-F1=0.5905

EVALUATE UCF | Attention-Only seed=42


Inference:   0%|          | 0/25 [00:00<?, ?it/s]

N=100 | AP=0.6486 | AUC=0.6568 | Macro-F1=0.5778

EVALUATE BV | Attention-Only seed=42


Inference:   0%|          | 0/350 [00:00<?, ?it/s]

N=1400 | AP=0.5243 | AUC=0.5365 | Macro-F1=0.5428

=== HELD-OUT RESULTS ===


,dataset,n,accuracy,balanced_accuracy,f1_macro,precision_violence,recall_violence,f1_violence,roc_auc,average_precision
0,RWF,1998,0.5921,0.5920,0.5905,0.6046,0.5301,0.5649,0.6231,0.6165
1,UCF,100,0.5900,0.5900,0.5778,0.6364,0.4200,0.5060,0.6568,0.6486
2,BV,1400,0.5429,0.5429,0.5428,0.5432,0.5386,0.5409,0.5365,0.5243


Saved: /content/drive/MyDrive/IVS/ViolenceDetection/Models/EffTempNet/HeldOut_Evaluation/AttentionOnly/seed_42/heldout_metrics_attention_only_seed_42.csv


## 8. Simpan evaluation manifest

Manifest menyimpan provenance evaluasi:
- checkpoint dan seed;
- preprocessing;
- konfigurasi BiLSTM;
- konfigurasi learned temporal attention;
- temporal aggregation;
- cache held-out yang digunakan.

In [13]:
manifest = {
    "task": "heldout_evaluation_only",
    "training_performed": False,
    "variant": VARIANT,
    "checkpoint_seed": RUN_SEED,
    "checkpoint_path": str(CHECKPOINT_PATH),
    "checkpoint_epoch": checkpoint.get("epoch"),
    "checkpoint_best_val_f1_macro": checkpoint.get("best_score"),
    "device": str(DEVICE),
    "torch_version": torch.__version__,
    "timm_version": timm.__version__,
    "sequence_length": SEQUENCE_LENGTH,
    "raw_resolution": [
        RAW_HEIGHT,
        RAW_WIDTH,
    ],
    "backbone_input_size": [
        BACKBONE_INPUT_SIZE,
        BACKBONE_INPUT_SIZE,
    ],
    "temporal_aggregation": "learned_attention_weighted_sum_of_bilstm_outputs",
    "uses_bilstm": True,
    "lstm_units": LSTM_UNITS,
    "bidirectional": True,
    "uses_temporal_attention": True,
    "attention_type": "TemporalAttention",
    "attention_projection_dim": 2 * LSTM_UNITS,
    "attention_normalization": "softmax_over_time",
    "normalization": "ImageNet",
    "use_amp": USE_AMP,
    "bootstrap_samples": BOOTSTRAP_SAMPLES,
    "bootstrap_seed": BOOTSTRAP_SEED,
    "heldout_datasets": list(
        HELDOUT_SPECS.keys()
    ),
    "checkpoint_config": checkpoint_config,
}

manifest_path = (
    EVAL_OUTPUT_DIR
    / "evaluation_manifest.json"
)

with open(
    manifest_path,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        manifest,
        f,
        indent=2,
        default=str,
    )

print("Manifest:", manifest_path)
print("\n✅ Attention-Only held-out evaluation selesai.")
print("✅ Tidak ada training/backpropagation pada notebook ini.")

Manifest: /content/drive/MyDrive/IVS/ViolenceDetection/Models/EffTempNet/HeldOut_Evaluation/AttentionOnly/seed_42/evaluation_manifest.json

✅ Attention-Only held-out evaluation selesai.
✅ Tidak ada training/backpropagation pada notebook ini.


# Seed berikutnya

Setelah evaluasi satu seed selesai, **jangan ubah preprocessing, dataset, cache,
atau arsitektur**.

Cukup kembali ke cell konfigurasi dan ubah satu baris:

```python
RUN_SEED = 123
```

menjadi:

```python
RUN_SEED = 0
```

atau:

```python
RUN_SEED = 42
```

Notebook akan otomatis:
- mencari `best.pt` Attention-Only untuk seed tersebut;
- memakai cache RWF/UCF/Bus yang sama;
- menulis hasil ke folder seed yang berbeda.

Jangan diubah antar-seed:

```python
VARIANT = "attention_only"
SEQUENCE_LENGTH = 16
LSTM_UNITS = 32
RAW_HEIGHT = 64
RAW_WIDTH = 64
BACKBONE_INPUT_SIZE = 224
EVAL_BATCH_SIZE = 4
```

Dengan demikian perbedaan hasil antar-seed berasal dari checkpoint training,
bukan dari perubahan sampling atau preprocessing held-out.